# Anatomy of a Redemption
### Spark-Based Multi-Source Sentiment Analysis of Cyberpunk 2077

Lab Group 1 — Data Processing 2, SS 2026

Batch pipeline: Reddit comments + Steam reviews over the 34-month window
(Dec 2020 – Dec 2023), sentiment via VADER, aligned against a patch timeline.
**Core thesis:** Reddit recovers faster than Steam.

**Stages:** 0 Config · 1 Ingest+Filter · 2 Cleaning · 3 VADER · 4 Aggregation · 5 Analysis/Plots

> Final, aggregated results go to `results/` (small, in the repo). The report is written from
> these files, not by re-running the pipeline.

## 0 · Configuration & Parameters (central)

All settings in **one block** — prerequisite for reproducibility. `DATA_MODE`
is the only switch between sample (development) and full run (`results/` snapshot). `SEED`
covers everything stochastic (especially LDA).

In [ ]:
import findspark
findspark.init()

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===== Central Parameters (one block — reproducibility) =====
DATA_MODE = "sample"            # "sample" | "full"
SEED = 42                       # for everything stochastic (especially LDA setSeed)
assert DATA_MODE in ("sample", "full")

WINDOW_START = "2020-12-01"     # 34-month window, inclusive bounds
WINDOW_END   = "2023-12-31"
SUBREDDITS   = ["cyberpunkgame", "LowSodiumCyberpunk", "Games", "pcgaming"]
STEAM_APP_ID = 1091500          # Cyberpunk 2077

ANOMALY_Z_THRESHOLD = 3.0       # volume anomaly / review-bombing flag
LDA_NUM_TOPICS = 8              # topic analysis

# ===== Paths =====
_SUB       = "sample" if DATA_MODE == "sample" else "raw"
RAW_REDDIT = f"data/{_SUB}/reddit/"
RAW_STEAM  = f"data/{_SUB}/steam/"
PROCESSED  = "data/processed/"
RESULTS    = "results/"
FIGURES    = "results/figures/"
# Patch timeline: small + manually maintained -> in the repo (data/sample/), applies to sample AND full
PATCH_TIMELINE = "data/sample/patch_timeline.csv"
Path(RESULTS).mkdir(exist_ok=True)
Path(FIGURES).mkdir(parents=True, exist_ok=True)

print(f"DATA_MODE={DATA_MODE}  window={WINDOW_START}..{WINDOW_END}  seed={SEED}")

### SparkSession
Local, `local[*]`. Driver RAM increased for the `full` run.

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CyberpunkSentiment")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "200")          # increase for full run if needed
    .config("spark.driver.memory", "8g" if DATA_MODE == "sample" else "12g")
    .config("spark.sql.files.maxPartitionBytes", "128m")    # controls partition size
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## 1 · Ingest + Immediate Filter

**Hard rule:** filter immediately on read (subreddits + time window), before anything
else happens. GB/month should become a fraction early. Spark reads `.zst` dumps directly.
Check column names (especially Steam) against the actual dataset before hardcoding.

Key idea: Spark never holds the full dataset in RAM — reads partitioned from disk, filters
immediately, writes a much smaller Parquet back. The same pipeline runs on sample as on the
full window.

In [ ]:
# -------------------------------------------------------------
# 1) Read Reddit (.zst, line-delimited JSON — Spark reads directly)
# -------------------------------------------------------------
reddit = (
    spark.read.json(RAW_REDDIT)                    # reads all files partitioned
    .select("subreddit", "body", "created_utc")    # only needed columns
    .filter(F.col("subreddit").isin(SUBREDDITS))   # filter to our subreddits
    .filter(F.col("body").isNotNull())
    .filter(F.col("body") != "[deleted]")
    .filter(F.col("body") != "[removed]")
)

# Time window (created_utc is Unix seconds)
START = F.unix_timestamp(F.lit(WINDOW_START), "yyyy-MM-dd")
END   = F.unix_timestamp(F.lit(WINDOW_END), "yyyy-MM-dd")
reddit = reddit.filter((F.col("created_utc") >= START) & (F.col("created_utc") <= END))

# Usable date + monthly bucket for later time series
reddit = (
    reddit
    .withColumn("date", F.to_date(F.from_unixtime("created_utc")))
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("reddit"))
    .withColumnRenamed("body", "text")
    .select("source", "month", "date", "text")
)

# -------------------------------------------------------------
# 2) Read Steam Reviews (CSV)
# -------------------------------------------------------------
steam = (
    spark.read.option("header", True).option("multiLine", True).csv(RAW_STEAM)
    .select(
        F.col("review").alias("text"),
        F.col("timestamp_created").cast("long").alias("created_utc"),
    )
    .filter(F.col("text").isNotNull())
)
steam = (
    steam
    .filter((F.col("created_utc") >= START) & (F.col("created_utc") <= END))
    .withColumn("date", F.to_date(F.from_unixtime("created_utc")))
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("steam"))
    .select("source", "month", "date", "text")
)

# -------------------------------------------------------------
# 3) Union + save as Parquet (partitioned by source + month)
# -------------------------------------------------------------
combined = reddit.unionByName(steam)

(
    combined.write
    .mode("overwrite")
    .partitionBy("source", "month")
    .parquet(PROCESSED + "cleaned/")
)
print("Cleaning done. Written to:", PROCESSED + "cleaned/")

# Verify at aggregated level (no collect/toPandas on the full df!)
combined.groupBy("source").count().show()

# -------------------------------------------------------------
# Continue from here with:  df = spark.read.parquet(PROCESSED + "cleaned/")
# This only loads the small, filtered data — fast.
# NEVER .collect() / .toPandas() on the full df! Aggregate first,
# THEN toPandas() on the tiny result for plots.
# -------------------------------------------------------------

### 1c · Patch Timeline (3rd Data Source)

Small, manually maintained table (`patch_name, release_date, platform, notes`) with the major
Cyberpunk 2077 patches in the window (1.0 Launch … 2.1). Stored in the repo under `data/sample/`.
Used later as patch markers on the sentiment time series and as anchor for the recovery metric.

In [ ]:
# -------------------------------------------------------------
# 4) Read Patch Timeline (3rd data source)
#    Small + manually maintained -> in the repo. Used later for
#    patch markers and the recovery metric (core thesis).
# -------------------------------------------------------------
patches = (
    spark.read.option("header", True).csv(PATCH_TIMELINE)
    .withColumn("release_date", F.to_date("release_date"))
    .withColumn("month", F.date_format("release_date", "yyyy-MM"))
    .filter(F.col("release_date").between(WINDOW_START, WINDOW_END))
)

# Small enough for toPandas (only ~9 rows) — used in Stage 5 for the overlay
patches.orderBy("release_date").show(truncate=False)

### Record Run Info
Which mode / which window the `results/` snapshot is based on (reproducibility).

In [ ]:
from datetime import datetime, timezone

Path(RESULTS, "RUN_INFO.txt").write_text(
    f"DATA_MODE = {DATA_MODE}\n"
    f"window    = {WINDOW_START} .. {WINDOW_END}\n"
    f"subreddits= {SUBREDDITS}\n"
    f"seed      = {SEED}\n"
    f"generated = {datetime.now(timezone.utc).isoformat()}\n"
)
print((Path(RESULTS) / "RUN_INFO.txt").read_text())

## 2 · Cleaning & Preprocessing  *(Phase 2 — TODO)*

- Normalize text (lowercasing, remove URLs/Markdown, whitespace)
- Language filter (English only — VADER is trained on English)
- Deduplication

## 3 · VADER Sentiment UDF  *(Phase 3 — TODO)*

VADER as Spark UDF on all comments/reviews; keep the `compound` score.

## 4 · Monthly/Source Aggregation  *(Phase 3 — TODO)*

Aggregate sentiment per `month` x `source` — **only here** does it become small enough for `.toPandas()`.

Writes: `results/sentiment_timeseries.csv` (`source, month, mean_sentiment, n`).

## 5 · Analysis & Visualization  *(Phase 3 — TODO)*

Each output -> exactly one file under `results/` (see report mapping in README):

- Two sentiment time series + patch markers  -> `results/figures/`, `results/patch_overlay.csv`
- Review-bombing anomaly flag (volume anomaly, `ANOMALY_Z_THRESHOLD`)  -> `results/anomaly_flags.csv`
- Sentiment without flagged windows  -> `results/sentiment_excl_bombing.csv`
- Recovery metric (core thesis)  -> `results/recovery_metric.csv`
- Topic analysis (TF-IDF / LDA with `setSeed(SEED)`, `LDA_NUM_TOPICS`)  -> `results/topics.csv`